# Automatización del Análisis Cualitativo de Comentarios NPS mediante Prompt Engineering

**Autor:** Cristian Javier Quijada Juárez
**Curso:** Inteligencia Artificial: Generación de Prompts - Comisión #96165
**Docentes:** Matías Galián · Javier Giménez

---

## Resumen

Este proyecto aborda un problema real del área de Experiencia de Cliente (CX): la plataforma Medallia recolecta miles de comentarios abiertos de clientes junto al puntaje de NPS (Net Promoter Score), pero ese texto libre - que contiene la causa raíz de la satisfacción o insatisfacción - hoy se revisa de forma manual, lenta y con sesgo de muestreo. Esta POC (proof of concept) demuestra, mediante técnicas de Fast Prompting sobre un modelo texto-texto (OpenAI API, `gpt-5.6-luna`), cómo automatizar la clasificación de comentarios por driver y sentimiento, iterando sobre el prompt en 3 versiones hasta lograr una clasificación precisa y con taxonomía estable, y la generación de una síntesis ejecutiva accionable. Complementariamente, se usa un modelo texto-imagen (Magnific AI) para traducir el hallazgo en una pieza de comunicación visual para stakeholders no técnicos.

## 1. Introducción

### 1.1 Nombre del proyecto
**Automatización del Análisis Cualitativo de Comentarios NPS mediante Prompt Engineering**

### 1.2 Presentación del problema

En Edenred, la plataforma Medallia recolecta el puntaje de NPS junto con comentarios abiertos de los clientes. El NPS mide qué tan probable es que un cliente recomiende el servicio, pero ese número por sí solo no explica *por qué* un cliente promotor, pasivo o detractor calificó así. Esa explicación vive en el comentario de texto libre, que hoy se revisa manualmente o con clasificaciones básicas por palabra clave.

Esto genera tres problemas concretos:

- **Tiempo:** revisar cientos o miles de comentarios manualmente no escala.
- **Sesgo de muestreo:** sin tiempo suficiente, solo se leen los comentarios más extremos, perdiendo señales intermedias relevantes.
- **Pérdida de accionabilidad:** los hallazgos cualitativos rara vez llegan estructurados y a tiempo a quienes toman decisiones.

Es relevante resolverlo porque el comentario abierto es la fuente más rica de causa raíz detrás del puntaje de NPS, y hoy está subutilizada.

### 1.3 Desarrollo de la propuesta de solución

La solución usa **prompt engineering sobre un modelo texto-texto** (OpenAI API, `gpt-5.6-luna`) para automatizar dos etapas del análisis cualitativo, y un **modelo texto-imagen** (Magnific AI) para comunicar el hallazgo:

- **Etapa 1 - Clasificación:** clasifica cada comentario por categoría de NPS, driver principal y sentimiento. Se desarrolla en **3 versiones iterativas** (v1, v2, v3), documentadas en detalle en la sección de Implementación.
- **Etapa 2 - Síntesis ejecutiva:** a partir de la clasificación final (v3), genera un resumen accionable para un comité no técnico.
- **Etapa 3 - Comunicación visual:** genera una infografía conceptual del hallazgo principal.

Un hallazgo clave de la Preentrega 1 (probado manualmente en ChatGPT) fue que el prompt original de clasificación confundía la *causa raíz* del contacto con el *manejo* del contacto (ej. un reclamo por pago rechazado se clasificaba como problema de "atención al cliente" en vez de "funcionalidad de la app"). En esta entrega, esa mejora se implementa vía API y se **compara empíricamente** contra la versión original - y, como se detalla más adelante, el primer intento de corrección introdujo un problema nuevo (categorías inventadas fuera de la taxonomía definida), que requirió una tercera iteración para resolverse. Este proceso de iteración es en sí mismo la evidencia central de aplicación de Fast Prompting que pide esta consigna.

### 1.4 Justificación de la viabilidad

- **Datos reales disponibles:** acceso directo a comentarios de Medallia en el rol actual del autor.
- **Costo marginal:** `gpt-5.6-luna` cuesta USD 0.20 / millón de tokens de entrada y USD 1.20 / millón de salida - para el volumen de este POC (menos de 20 comentarios cortos), el costo total de las 4 llamadas realizadas en esta notebook es de fracciones de centavo.
- **Herramientas ya en uso:** OpenAI API (tier gratuito) y Magnific AI (suscripción activa del autor) - sin inversión adicional.
- **Número de llamadas a la API, justificado:** esta notebook realiza **4 llamadas** en total: 3 versiones del prompt de clasificación (v1, v2, v3) para documentar el proceso de iteración de Fast Prompting, y 1 llamada de síntesis que reutiliza el resultado ya clasificado de v3 sin reprocesar los comentarios crudos. No se realizan llamadas redundantes: cada una aporta evidencia distinta al análisis.

## 2. Objetivos

- Automatizar la clasificación de comentarios abiertos de NPS por categoría, driver de insatisfacción/satisfacción y sentimiento, usando un modelo de lenguaje vía API.
- Iterar sobre el diseño del prompt de clasificación, documentando cada corrección y su efecto medible, para demostrar mejora real en la precisión y estabilidad de la clasificación.
- Generar una síntesis ejecutiva accionable a partir de la clasificación ya validada, sin reprocesar los datos crudos.
- Producir una pieza de comunicación visual del hallazgo mediante un modelo texto-imagen.
- Mantener el proyecto rentable: justificar cada consulta a la API y evitar llamadas redundantes.

## 3. Metodología

El proyecto se desarrolla en 3 etapas secuenciales, cada una construida sobre el resultado de la anterior para evitar procesamiento redundante:

1. **Preparación de datos:** se define un set de 8 comentarios de ejemplo (representativos de las 3 categorías de NPS), simulando un extracto real de Medallia.
2. **Clasificación con Fast Prompting, iterada en 3 versiones (Etapa 1):**
   - **v1 (línea base):** el prompt original, tal como se probó manualmente en la Preentrega 1, que presentó una ambigüedad de clasificación conocida (confundir causa raíz del contacto con calidad de atención).
   - **v2 (primera corrección - constraint prompting):** se agrega una regla que distingue explícitamente "causa raíz del contacto" de "calidad de la atención recibida". Corrige el error de v1, pero al ejecutarla vía API se detectó un efecto secundario no anticipado: el modelo empezó a inventar categorías de driver nuevas y más granulares (ej. "pago rechazado", "cargo duplicado"), fuera de la lista de categorías originalmente sugerida.
   - **v3 (segunda corrección - constraint + lista cerrada obligatoria):** se mantiene la regla de v2 y se agrega una restricción adicional que obliga al modelo a elegir únicamente entre 5 categorías fijas, sin inventar variantes. Esta versión corrige el error original de v1 **y** mantiene la taxonomía estable, condición necesaria para que la clasificación sea comparable entre ciclos de medición en un uso real.
   Se comparan las 3 versiones fila por fila para verificar el efecto de cada iteración.
3. **Síntesis ejecutiva (Etapa 2):** el resultado de v3 (ya validado como preciso y con taxonomía estable) se pasa como contexto de entrada al prompt de síntesis, **sin volver a enviar los comentarios crudos ni repetir la clasificación**.
4. **Comunicación visual (Etapa 3):** el prompt de imagen se ejecuta manualmente en Magnific AI (no vía API, según lo permite la consigna cuando no se usa Dall-E), y el resultado se documenta con el prompt exacto utilizado y la imagen obtenida.

Cada paso se justifica por su aporte al objetivo y se documenta con su costo/beneficio, en línea con la recomendación de evaluar la rentabilidad de las consultas a la API.

## 4. Herramientas y tecnologías

- **Modelo texto-texto:** OpenAI API, modelo `gpt-5.6-luna` - elegido por ser el modelo de menor costo de la familia GPT-5.6, orientado explícitamente a cargas de trabajo de clasificación y automatización de alto volumen y sensibles al costo. Usar un modelo más grande sería sobre-ingeniería para una tarea de clasificación de texto corto. **Nota técnica:** este modelo no admite modificar el parámetro `temperature` (solo acepta su valor por defecto), a diferencia de generaciones anteriores de modelos OpenAI - se ajustó el código para omitir este parámetro tras detectar el error en ejecución.
- **Modelo texto-imagen:** Magnific AI (anteriormente Freepik, con generador Mystic integrado) - usado manualmente (sin API) según lo habilitado por la consigna al no usar Dall-E.
- **Técnicas de Fast Prompting aplicadas:**
  - **Role prompting:** se le asigna al modelo un rol específico ("analista de experiencia de cliente especializado en NPS") para acotar el estilo y criterio de respuesta.
  - **Structured output / output formatting:** se exige una salida en formato JSON estricto, eliminando texto libre adicional - esto reduce tokens de salida (menor costo) y hace el resultado directamente parseable en código, sin necesidad de post-procesamiento manual.
  - **Constraint prompting (aplicado en v2 y v3):** se agrega una regla explícita que fuerza al modelo a distinguir causa raíz del contacto vs. calidad de atención, corrigiendo la ambigüedad detectada en v1.
  - **Restricción de vocabulario cerrado (aplicado en v3):** a diferencia de v2 (que solo daba categorías "de ejemplo"), v3 obliga explícitamente a elegir entre una lista fija de 5 opciones, sin variantes - esta es la técnica que corrigió el efecto secundario detectado en v2.
  - **Prompt chaining sin llamadas redundantes:** el output estructurado (JSON) de la Etapa 1 (v3) se reutiliza como input textual de la Etapa 2, evitando una llamada adicional de reprocesamiento.

## 5. Implementación

### 5.1 Configuración del entorno

Esta notebook está preparada para ejecutarse en **Google Colab**. La API key se lee desde **Colab Secrets** (panel de la llave 🔑 a la izquierda), bajo el nombre `OPENAI_API_KEY`, evitando escribir la key en cualquier archivo del proyecto.

Si se ejecuta fuera de Colab (entorno local), reemplazar la celda siguiente por la carga desde un archivo `.env` (ver `.env.example` en el repositorio) usando `python-dotenv`.

In [9]:
from google.colab import userdata
import json
from openai import OpenAI

api_key = userdata.get('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError(
        "No se encontró OPENAI_API_KEY en Colab Secrets. Agrégalo desde el panel de la llave."
    )

client = OpenAI(api_key=api_key)
MODEL = "gpt-5.6-luna"

# Contador manual de llamadas a la API, para verificar rentabilidad (requisito de la consigna)
api_calls_made = 0

print("Cliente configurado correctamente. Modelo:", MODEL)

Cliente configurado correctamente. Modelo: gpt-5.6-luna


### 5.2 Datos de entrada

Set de 8 comentarios de ejemplo, representativos de las 3 categorías de NPS (Promotor, Pasivo, Detractor), simulando un extracto real de Medallia.

In [10]:
comentarios = [
    {"id": 1, "nps": 9,  "texto": "La app es muy fácil de usar y el saldo se actualiza rápido. Ya la recomendé a dos compañeros de trabajo."},
    {"id": 2, "nps": 3,  "texto": "Llevo dos semanas intentando que me resuelvan un problema de pago rechazado y nadie me da una solución clara. Muy mal servicio."},
    {"id": 3, "nps": 7,  "texto": "En general funciona bien, pero me gustaría que hubiera más comercios afiliados cerca de mi zona."},
    {"id": 4, "nps": 10, "texto": "Excelente atención, el asesor fue muy amable y resolvió mi duda en minutos. Todo perfecto."},
    {"id": 5, "nps": 2,  "texto": "La aplicación se traba constantemente al momento de pagar en el punto de venta, es muy frustrante y me ha hecho quedar mal."},
    {"id": 6, "nps": 6,  "texto": "El servicio cumple, pero el tiempo de espera en el chat de soporte es demasiado largo, más de 30 minutos a veces."},
    {"id": 7, "nps": 8,  "texto": "Buena cobertura de comercios y el proceso de activación de la tarjeta fue sencillo."},
    {"id": 8, "nps": 1,  "texto": "Tuve un cargo duplicado y hasta ahora, después de tres llamadas, sigue sin resolverse. Pésima experiencia."},
]

def formatear_comentarios(lista):
    return "\n".join(f"ID {c['id']} (NPS {c['nps']}): {c['texto']}" for c in lista)

for c in comentarios:
    print(f"ID {c['id']} (NPS {c['nps']}): {c['texto']}")

ID 1 (NPS 9): La app es muy fácil de usar y el saldo se actualiza rápido. Ya la recomendé a dos compañeros de trabajo.
ID 2 (NPS 3): Llevo dos semanas intentando que me resuelvan un problema de pago rechazado y nadie me da una solución clara. Muy mal servicio.
ID 3 (NPS 7): En general funciona bien, pero me gustaría que hubiera más comercios afiliados cerca de mi zona.
ID 4 (NPS 10): Excelente atención, el asesor fue muy amable y resolvió mi duda en minutos. Todo perfecto.
ID 5 (NPS 2): La aplicación se traba constantemente al momento de pagar en el punto de venta, es muy frustrante y me ha hecho quedar mal.
ID 6 (NPS 6): El servicio cumple, pero el tiempo de espera en el chat de soporte es demasiado largo, más de 30 minutos a veces.
ID 7 (NPS 8): Buena cobertura de comercios y el proceso de activación de la tarjeta fue sencillo.
ID 8 (NPS 1): Tuve un cargo duplicado y hasta ahora, después de tres llamadas, sigue sin resolverse. Pésima experiencia.


### 5.3 Etapa 1 - Clasificación (v1: línea base)

Esta es la versión del prompt tal como se probó manualmente en la Preentrega 1. Se ejecuta aquí vía API para tener una línea base medible antes de aplicar mejoras.

**Nota técnica:** el modelo `gpt-5.6-luna` no admite el parámetro `temperature` con valores distintos al default, por lo que se omite en todas las llamadas de esta notebook (a diferencia de versiones anteriores de este prompt, probadas en la interfaz de ChatGPT).

In [11]:
PROMPT_V1 = '''Actúa como analista de experiencia de cliente especializado en NPS.
Te compartiré un lote de comentarios abiertos de clientes, cada uno con su puntaje de NPS asociado (0-10).

Para cada comentario, devuelve un JSON con una lista de objetos, cada uno con las claves:
- "id": ID del comentario
- "clasificacion": "Promotor" (9-10) / "Pasivo" (7-8) / "Detractor" (0-6)
- "driver": driver principal mencionado (ej: tiempos de atencion, funcionalidad de la app, cobertura de comercios, atencion al cliente, otro)
- "sentimiento": "positivo" / "negativo" / "neutro"

Responde ÚNICAMENTE con el JSON, sin texto adicional.

Comentarios:
{comentarios}
'''

respuesta_v1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": PROMPT_V1.format(comentarios=formatear_comentarios(comentarios))}
    ],
)
api_calls_made += 1

resultado_v1 = json.loads(respuesta_v1.choices[0].message.content)
print(json.dumps(resultado_v1, indent=2, ensure_ascii=False))

[
  {
    "id": 1,
    "clasificacion": "Promotor",
    "driver": "funcionalidad de la app",
    "sentimiento": "positivo"
  },
  {
    "id": 2,
    "clasificacion": "Detractor",
    "driver": "atencion al cliente",
    "sentimiento": "negativo"
  },
  {
    "id": 3,
    "clasificacion": "Pasivo",
    "driver": "cobertura de comercios",
    "sentimiento": "neutro"
  },
  {
    "id": 4,
    "clasificacion": "Promotor",
    "driver": "atencion al cliente",
    "sentimiento": "positivo"
  },
  {
    "id": 5,
    "clasificacion": "Detractor",
    "driver": "funcionalidad de la app",
    "sentimiento": "negativo"
  },
  {
    "id": 6,
    "clasificacion": "Detractor",
    "driver": "tiempos de atencion",
    "sentimiento": "negativo"
  },
  {
    "id": 7,
    "clasificacion": "Pasivo",
    "driver": "cobertura de comercios",
    "sentimiento": "positivo"
  },
  {
    "id": 8,
    "clasificacion": "Detractor",
    "driver": "atencion al cliente",
    "sentimiento": "negativo"
  }
]


### 5.4 Etapa 1 - Clasificación (v2: constraint prompting)

Se aplica **constraint prompting**: se agrega una regla explícita que obliga al modelo a distinguir la causa raíz del contacto de la calidad de la atención recibida - la ambigüedad detectada como hallazgo en la Preentrega 1 (el comentario del ID 2 se clasificaba incorrectamente como "atención al cliente" cuando la causa real era un fallo de pago).

In [12]:
PROMPT_V2 = '''Actúa como analista de experiencia de cliente especializado en NPS.
Te compartiré un lote de comentarios abiertos de clientes, cada uno con su puntaje de NPS asociado (0-10).

Para cada comentario, devuelve un JSON con una lista de objetos, cada uno con las claves:
- "id": ID del comentario
- "clasificacion": "Promotor" (9-10) / "Pasivo" (7-8) / "Detractor" (0-6)
- "driver": driver principal mencionado (ej: tiempos de atencion, funcionalidad de la app, cobertura de comercios, atencion al cliente, otro)
- "sentimiento": "positivo" / "negativo" / "neutro"

REGLA IMPORTANTE: distingue explícitamente entre la CAUSA RAÍZ del contacto
(el problema técnico u operativo que originó la queja, ej. pago rechazado,
cargo duplicado, falla de la app) y la CALIDAD DE LA ATENCIÓN recibida
(cómo fue tratado el cliente por un asesor humano). Un comentario debe
clasificarse como "atencion al cliente" ÚNICAMENTE si la queja o elogio
es sobre el trato o la gestión del asesor, NO si menciona que un
problema técnico "no fue resuelto" - en ese caso, el driver es el
problema técnico subyacente.

Responde ÚNICAMENTE con el JSON, sin texto adicional.

Comentarios:
{comentarios}
'''

respuesta_v2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": PROMPT_V2.format(comentarios=formatear_comentarios(comentarios))}
    ],
)
api_calls_made += 1

resultado_v2 = json.loads(respuesta_v2.choices[0].message.content)
print(json.dumps(resultado_v2, indent=2, ensure_ascii=False))

[
  {
    "id": 1,
    "clasificacion": "Promotor",
    "driver": "funcionalidad de la app",
    "sentimiento": "positivo"
  },
  {
    "id": 2,
    "clasificacion": "Detractor",
    "driver": "pago rechazado",
    "sentimiento": "negativo"
  },
  {
    "id": 3,
    "clasificacion": "Pasivo",
    "driver": "cobertura de comercios",
    "sentimiento": "neutro"
  },
  {
    "id": 4,
    "clasificacion": "Promotor",
    "driver": "atencion al cliente",
    "sentimiento": "positivo"
  },
  {
    "id": 5,
    "clasificacion": "Detractor",
    "driver": "falla de la app",
    "sentimiento": "negativo"
  },
  {
    "id": 6,
    "clasificacion": "Detractor",
    "driver": "tiempos de atencion",
    "sentimiento": "negativo"
  },
  {
    "id": 7,
    "clasificacion": "Pasivo",
    "driver": "cobertura de comercios",
    "sentimiento": "positivo"
  },
  {
    "id": 8,
    "clasificacion": "Detractor",
    "driver": "cargo duplicado",
    "sentimiento": "negativo"
  }
]


**Hallazgo tras ejecutar v2 vía API:** el resultado real mostró que el modelo sí corrigió el ID 2 y el ID 8 (ya no los clasifica como "atención al cliente"), pero introdujo un efecto secundario no anticipado: en vez de usar las categorías sugeridas como ejemplos, inventó categorías nuevas y más específicas en 3 de los 8 comentarios ("pago rechazado" para el ID 2, "falla de la app" para el ID 5, y "cargo duplicado" para el ID 8) - incluyendo el ID 5, que ya estaba bien clasificado en v1 y no necesitaba cambiar. Esto es un problema real para un caso de uso en producción: si la taxonomía de drivers cambia en cada corrida, incluso en casos que ya eran correctos, no es posible comparar resultados de forma consistente entre ciclos de medición de NPS. Este hallazgo motivó la versión v3.

### 5.5 Etapa 1 - Clasificación (v3: constraint prompting + vocabulario cerrado)

Se mantiene la regla de distinción de causa raíz de v2, y se agrega una restricción adicional: el modelo debe elegir **obligatoriamente** entre una lista cerrada de 5 categorías, sin inventar variantes nuevas. Esta es la técnica de Fast Prompting que resuelve el efecto secundario detectado en v2.

In [13]:
PROMPT_V3 = '''Actúa como analista de experiencia de cliente especializado en NPS.
Te compartiré un lote de comentarios abiertos de clientes, cada uno con su puntaje de NPS asociado (0-10).

Para cada comentario, devuelve un JSON con una lista de objetos, cada uno con las claves:
- "id": ID del comentario
- "clasificacion": "Promotor" (9-10) / "Pasivo" (7-8) / "Detractor" (0-6)
- "driver": debes elegir OBLIGATORIAMENTE una de estas 5 opciones exactas, sin
  inventar categorías nuevas ni usar variantes: "tiempos de atencion",
  "funcionalidad de la app", "cobertura de comercios", "atencion al cliente", "otro"
- "sentimiento": "positivo" / "negativo" / "neutro"

REGLA IMPORTANTE: distingue explícitamente entre la CAUSA RAÍZ del contacto
(el problema técnico u operativo que originó la queja, ej. pago rechazado,
cargo duplicado, falla de la app) y la CALIDAD DE LA ATENCIÓN recibida
(cómo fue tratado el cliente por un asesor humano). Un comentario debe
clasificarse como "atencion al cliente" ÚNICAMENTE si la queja o elogio
es sobre el trato o la gestión del asesor, NO si menciona que un
problema técnico "no fue resuelto" - en ese caso, el driver es
"funcionalidad de la app" (para fallas de pago, cargos duplicados, errores
de la app) o "otro" si no encaja en ninguna categoría de la lista.

Responde ÚNICAMENTE con el JSON, sin texto adicional.

Comentarios:
{comentarios}
'''

respuesta_v3 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": PROMPT_V3.format(comentarios=formatear_comentarios(comentarios))}
    ],
)
api_calls_made += 1

resultado_v3 = json.loads(respuesta_v3.choices[0].message.content)
print(json.dumps(resultado_v3, indent=2, ensure_ascii=False))

[
  {
    "id": 1,
    "clasificacion": "Promotor",
    "driver": "funcionalidad de la app",
    "sentimiento": "positivo"
  },
  {
    "id": 2,
    "clasificacion": "Detractor",
    "driver": "funcionalidad de la app",
    "sentimiento": "negativo"
  },
  {
    "id": 3,
    "clasificacion": "Pasivo",
    "driver": "cobertura de comercios",
    "sentimiento": "neutro"
  },
  {
    "id": 4,
    "clasificacion": "Promotor",
    "driver": "atencion al cliente",
    "sentimiento": "positivo"
  },
  {
    "id": 5,
    "clasificacion": "Detractor",
    "driver": "funcionalidad de la app",
    "sentimiento": "negativo"
  },
  {
    "id": 6,
    "clasificacion": "Detractor",
    "driver": "tiempos de atencion",
    "sentimiento": "negativo"
  },
  {
    "id": 7,
    "clasificacion": "Pasivo",
    "driver": "cobertura de comercios",
    "sentimiento": "positivo"
  },
  {
    "id": 8,
    "clasificacion": "Detractor",
    "driver": "funcionalidad de la app",
    "sentimiento": "negativo"
  }
]


### 5.6 Comparación v1 vs v2 vs v3 (evidencia de iteración de Fast Prompting)

Se comparan las 3 versiones fila por fila, con foco en los IDs 2 y 8 (los casos que cambiaron entre versiones).

In [14]:
import pandas as pd

df_v1 = pd.DataFrame(resultado_v1).set_index("id").add_suffix("_v1")
df_v2 = pd.DataFrame(resultado_v2).set_index("id").add_suffix("_v2")
df_v3 = pd.DataFrame(resultado_v3).set_index("id").add_suffix("_v3")

comparacion = df_v1.join(df_v2).join(df_v3)[
    ["driver_v1", "driver_v2", "driver_v3", "clasificacion_v1", "clasificacion_v2", "clasificacion_v3"]
]
comparacion

,driver_v1,driver_v2,driver_v3,clasificacion_v1,clasificacion_v2,clasificacion_v3
id,,,,,,
1,funcionalidad de la app,funcionalidad de la app,funcionalidad de la app,Promotor,Promotor,Promotor
2,atencion al cliente,pago rechazado,funcionalidad de la app,Detractor,Detractor,Detractor
3,cobertura de comercios,cobertura de comercios,cobertura de comercios,Pasivo,Pasivo,Pasivo
4,atencion al cliente,atencion al cliente,atencion al cliente,Promotor,Promotor,Promotor
5,funcionalidad de la app,falla de la app,funcionalidad de la app,Detractor,Detractor,Detractor
6,tiempos de atencion,tiempos de atencion,tiempos de atencion,Detractor,Detractor,Detractor
7,cobertura de comercios,cobertura de comercios,cobertura de comercios,Pasivo,Pasivo,Pasivo
8,atencion al cliente,cargo duplicado,funcionalidad de la app,Detractor,Detractor,Detractor


**Lectura del resultado:**

- **ID 2:** `v1 = "atencion al cliente"` (incorrecto) → `v2 = "pago rechazado"` (corregido, pero taxonomía inventada) → `v3 = "funcionalidad de la app"` (corregido **y** dentro de la lista cerrada).
- **ID 5:** `v1 = "funcionalidad de la app"` (ya correcto) → `v2 = "falla de la app"` (el modelo reformuló una categoría que ya era correcta, generando una variante de redacción no prevista) → `v3 = "funcionalidad de la app"` (vuelve a la categoría exacta de la lista cerrada).
- **ID 8:** mismo patrón que el ID 2 - `v1 = "atencion al cliente"` (incorrecto) → `v2 = "cargo duplicado"` (taxonomía inventada) → `v3 = "funcionalidad de la app"` (correcto y estable).
- En total, **3 de los 8 comentarios (IDs 2, 5 y 8)** mostraron inconsistencia de taxonomía en v2 - no solo los dos casos con error de causa raíz, sino también un caso que ya estaba bien clasificado en v1 y que v2 reformuló innecesariamente. Esto refuerza el hallazgo: sin una restricción de vocabulario cerrado, el modelo tiende a generar variantes de redacción libre incluso en categorías que no necesitaban corrección.
- El resto de los IDs se mantiene estable entre las 3 versiones.

Esta progresión (v1 con error → v2 corrige el error de causa raíz pero introduce inconsistencia de taxonomía en 3 de 8 casos → v3 corrige ambos problemas) es la evidencia empírica de que **iterar sobre el prompt con distintas técnicas de Fast Prompting mejora medible y verificablemente la propuesta de solución planteada en la Preentrega 1**.

### 5.7 Etapa 2 - Síntesis ejecutiva

Esta etapa reutiliza el resultado de **v3** (la versión validada como precisa y con taxonomía estable) como parte del prompt. No se vuelve a enviar el texto crudo de los comentarios ni se re-clasifica nada - esto evita una llamada redundante a la API.

In [15]:
PROMPT_SINTESIS = '''Actúa como consultor de CX presentando hallazgos a un comité directivo no técnico.

Con base en esta clasificación de comentarios (JSON con id, clasificacion, driver y sentimiento):

{clasificacion}

Genera un resumen ejecutivo de máximo 200 palabras que incluya:
1. El driver de insatisfacción más frecuente entre los Detractores.
2. Una comparación breve con lo que valoran los Promotores.
3. Una recomendación accionable priorizada (qué atacar primero y por qué).

Usa lenguaje claro, sin jerga técnica de ciencia de datos.
'''

respuesta_sintesis = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": PROMPT_SINTESIS.format(
            clasificacion=json.dumps(resultado_v3, ensure_ascii=False)
        )}
    ],
)
api_calls_made += 1

sintesis = respuesta_sintesis.choices[0].message.content
print(sintesis)

### Resumen ejecutivo

La **funcionalidad de la app** es el principal foco de insatisfacción: aparece en **3 de los 4 comentarios de los Detractores (75%)**. En segundo lugar se encuentran los **tiempos de atención**, mencionados una vez.

Los Promotores valoran tanto la **funcionalidad de la app** como la **atención al cliente**, lo que indica que la experiencia digital tiene potencial para generar satisfacción, pero actualmente presenta una ejecución inconsistente. También se observan opiniones neutrales sobre la cobertura de comercios, sin señales de urgencia comparable.

**Recomendación priorizada:** atacar primero la funcionalidad de la app. Se debe identificar y corregir los principales problemas que generan frustración —por ejemplo, errores, dificultades de uso o fallas en procesos clave— y validar las mejoras con clientes. Esta prioridad se justifica porque concentra la mayoría de las experiencias negativas y, al mismo tiempo, ya es un atributo valorado por algunos Promotores. 

In [16]:
print(f"Total de llamadas realizadas a la API en esta notebook: {api_calls_made}")

Total de llamadas realizadas a la API en esta notebook: 4


**Nota de rentabilidad:** esta notebook realiza 4 llamadas en total (v1, v2, v3, síntesis). Las primeras 3 no son redundantes entre sí: cada una documenta una iteración distinta del prompt de clasificación, evidencia central de Fast Prompting que pide esta consigna. La llamada de síntesis reutiliza directamente el JSON de v3 ya en memoria, sin reenviar los comentarios crudos ni repetir la clasificación - ahí es donde se aplicó la optimización de costos real.

### 5.8 Etapa 3 - Comunicación visual (texto-imagen)

Generado manualmente en **Magnific AI** (no vía API, conforme lo permite la consigna al no usar Dall-E). Se documenta el prompt exacto utilizado y el resultado obtenido.

**Prompt utilizado:**

```
Infografía corporativa minimalista para presentación ejecutiva, tema
"Voz del Cliente". Ilustra el concepto de un comentario de cliente
transformándose en un insight accionable para el negocio. Paleta de
colores corporativos azul y blanco, estilo flat design, iconografía de
burbujas de diálogo y gráfico de tendencia ascendente, sin texto dentro
de la imagen, composición limpia apta para slide de PowerPoint, alta
resolución, fondo neutro.
```

**Resultado obtenido:**

![Resultado Magnific AI](images/prompt3_resultado.png)

**Hallazgo del proceso:** la primera ejecución del prompt ignoró la instrucción "sin texto dentro de la imagen" y generó una infografía completa con títulos y etiquetas en inglés, no utilizable para un comité en español sin edición. Al reintentar, el modelo sí respetó la restricción y entregó la pieza limpia mostrada arriba. Esto confirma que el resultado del modelo texto-imagen es inconsistente entre corridas - una limitación técnica real que se documenta en la sección de Resultados.

## 6. Resultados

- **Etapa 1 (clasificación):** se identificó y corrigió, mediante 3 iteraciones de Fast Prompting, tanto el error de causa raíz (v1 → v2) como un efecto secundario no anticipado de taxonomía inconsistente (v2 → v3). La versión final (v3) corrige ambos problemas de forma verificable en la tabla comparativa de la sección 5.6.
- **Etapa 2 (síntesis):** el resumen ejecutivo generado es claro, breve y accionable, cumpliendo el límite de 200 palabras y el lenguaje no técnico solicitado, y se basa en la clasificación más precisa y estable disponible (v3).
- **Etapa 3 (imagen):** se obtuvo una pieza visual utilizable, aunque con una limitación de consistencia entre corridas que debe considerarse al llevar esto a producción (puede requerir 2-3 intentos).
- **Costo:** la notebook realizó 4 llamadas a la API en total - 3 para documentar el proceso de iteración de clasificación y 1 para la síntesis - sin ninguna llamada redundante de reprocesamiento de datos crudos.

En conjunto, la implementación **sí logra la solución esperada**: automatiza la clasificación y síntesis de comentarios abiertos de NPS, con evidencia medible y documentada de mejora vía Fast Prompting, incluyendo la corrección de un problema que no era evidente hasta ejecutar el código real (la inconsistencia de taxonomía en v2).

## 7. Conclusiones

- Se cumplieron los objetivos planteados: se demostró comprensión y aplicación de técnicas de Fast Prompting (role prompting, structured output, constraint prompting, restricción de vocabulario cerrado), se experimentó con 3 configuraciones distintas de prompt para optimizar la eficacia, y se preparó una demostración funcional en Jupyter Notebook ejecutada en Google Colab.
- El hallazgo de la Preentrega 1 (ambigüedad de clasificación en el ID 2) no solo se documentó, sino que se **resolvió y verificó empíricamente** en esta entrega - y, en el proceso, se descubrió y corrigió un segundo problema (taxonomía inconsistente en v2) que no era evidente durante las pruebas manuales de la entrega anterior. Esto demuestra que ejecutar el prompt vía código, con datos estructurados, revela problemas que las pruebas manuales en la interfaz de chat no siempre exponen.
- La limitación de consistencia del modelo texto-imagen (Magnific AI) queda como un punto abierto a mitigar en una futura iteración, por ejemplo, generando 2-3 variantes por defecto y seleccionando la más adecuada.
- El diseño de la solución prioriza la rentabilidad: cada una de las 4 llamadas a la API realizadas está justificada (3 iteraciones de clasificación documentadas + 1 síntesis), sin llamadas de reprocesamiento innecesarias.
- Como siguiente paso natural (fuera del alcance de esta entrega), valdría la pena probar el pipeline con un volumen real de comentarios de Medallia y medir si la precisión y estabilidad de taxonomía de v3 se mantienen fuera del set de prueba controlado.

## 8. Referencias

- Documentación oficial de modelos OpenAI: https://developers.openai.com/api/docs/models
- Plataforma Magnific AI (generación de imagen): https://www.magnific.com/app
- Preentrega 1 de este mismo proyecto (documento base del análisis del problema y primeros prompts probados manualmente).